# Testing - PI Data Engineering Challenge

Notebook de validación para comprobar estructura, volumen, calidad de datos, duplicados y posibles pérdidas de información.

### Criterio del challenge
Un registro se considera duplicado cuando `[ID]`, `[MUESTRA]` y `[RESULTADO]` son iguales. En caso de duplicados se debe conservar el último registro copiado.

> **Importante:** completar las variables de configuración de la siguiente celda con los nombres reales de tus tablas.

In [0]:
# ==============================
# 1. CONFIGURACIÓN
# ==============================

# Tabla con los datos de entrada del proceso.
SOURCE_TABLE = "challenge.bronze_catalog.unificado"

# Tabla final después de la carga/deduplicación.
TARGET_TABLE = "challenge.silver_catalog.unificado_final"

# Clave de negocio definida por el challenge.
KEY_COLUMNS = ["ID", "MUESTRA", "RESULTADO"]

# Columna utilizada para determinar cuál es el último registro.
DATE_COLUMN = "FECHA_COPIA"

print(f"Fuente : {SOURCE_TABLE}")
print(f"Destino : {TARGET_TABLE}")
print(f"Claves   : {KEY_COLUMNS}")

## 2. Existencia y estructura de las tablas

In [0]:
# Verificar que ambas tablas existan.
source_exists = spark.catalog.tableExists(SOURCE_TABLE)
target_exists = spark.catalog.tableExists(TARGET_TABLE)

display(spark.createDataFrame([
    (SOURCE_TABLE, source_exists),
    (TARGET_TABLE, target_exists)
], ["table", "exists"]))

assert source_exists, f"No existe la tabla origen: {SOURCE_TABLE}"
assert target_exists, f"No existe la tabla destino: {TARGET_TABLE}"

In [0]:
# Mostrar estructura de las tablas.
print("=== SOURCE ===")
spark.table(SOURCE_TABLE).printSchema()

print("\n=== TARGET ===")
spark.table(TARGET_TABLE).printSchema()

In [0]:
# Comparar nombres y tipos de columnas.
source_schema = spark.table(SOURCE_TABLE).schema
target_schema = spark.table(TARGET_TABLE).schema

source_fields = {f.name.upper(): f.dataType.simpleString() for f in source_schema.fields}
target_fields = {f.name.upper(): f.dataType.simpleString() for f in target_schema.fields}

all_columns = sorted(set(source_fields) | set(target_fields))

schema_rows = []
for c in all_columns:
    s = source_fields.get(c)
    t = target_fields.get(c)
    schema_rows.append((c, s, t, s == t))

schema_df = spark.createDataFrame(
    schema_rows,
    ["columna", "fuente_tipo", "destino_tipo", "tipo_match"]
)
display(schema_df)

schema_issues = schema_df.filter("fuente_tipo IS NULL OR destino_tipo IS NULL OR tipo_match = false").count()
print(f"Schema issues: {schema_issues}")

## 3. Validar que existan las columnas clave en el destino

In [0]:
target_columns = {c.name.upper() for c in spark.table(TARGET_TABLE).schema.fields}
missing_keys = [c for c in KEY_COLUMNS if c.upper() not in target_columns]

if missing_keys:
    raise Exception(f"Faltan columnas clave en silver.unificado_final: {missing_keys}")

if DATE_COLUMN.upper() not in target_columns:
    raise Exception(f"Falta la columna {DATE_COLUMN} en silver.unificado_final")

print("OK - Las columnas clave y FECHA_COPIA existen en silver.unificado_final")

## 4. Comparación de volumen

In [0]:
source_df = spark.table(SOURCE_TABLE)
target_df = spark.table(TARGET_TABLE)

source_count = source_df.count()
target_count = target_df.count()
source_distinct_keys = source_df.select(*KEY_COLUMNS).dropDuplicates().count()
target_distinct_keys = target_df.select(*KEY_COLUMNS).dropDuplicates().count()

volume_df = spark.createDataFrame([
    ("SOURCE total rows", source_count),
    ("TARGET total rows", target_count),
    ("SOURCE distinct business keys", source_distinct_keys),
    ("TARGET distinct business keys", target_distinct_keys),
], ["metric", "value"])
display(volume_df)

### Interpretación
- Si el origen contiene duplicados, `bronze.unificado total filas` puede ser mayor que `silver.unificado_final total filas`.
- Para este challenge, una comprobación importante es que la cantidad de **claves de negocio distintas** se conserve.
- `bronze.unificado distinct claves de negocio` y `silver.unificado_final distinct claves de negocio` deberían coincidir cuando el destino contiene todos los registros procesados.

## 5. Valores NULL por campo

In [0]:
from pyspark.sql import functions as F

def null_report(df, table_name):
    total = df.count()
    expressions = []
    for c in df.columns:
        expressions.append(
            F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        )
    row = df.agg(*expressions).collect()[0].asDict()
    rows = [
        (table_name, c, int(row[c] or 0), total, round((int(row[c] or 0) / total * 100) if total else 0.0, 2))
        for c in df.columns
    ]
    return spark.createDataFrame(rows, ["tabla", "columna", "conteo nulos", "total_filas", "porcentaje nulos"])

nulls_source = null_report(source_df, "bronze.unificado")
nulls_target = null_report(target_df, "silver.unificado_final")

nulls = nulls_source.unionByName(nulls_target)
display(nulls.orderBy("columna", "tabla"))

## 6. Validar NULLs en las columnas clave

In [0]:
for c in KEY_COLUMNS:
    source_nulls = source_df.filter(F.col(c).isNull()).count()
    target_nulls = target_df.filter(F.col(c).isNull()).count()
    print(f"{c}: bronze.unificado NULLs={source_nulls} | silver.unificado_final  NULLs={target_nulls}")

## 7. Detectar duplicados en silver_unificado_final 

In [0]:
duplicates_target = (
    target_df
    .groupBy(*KEY_COLUMNS)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_groups = duplicates_target.count()
print(f"Cantidad de grupos duplicados en silver.unificado_final : {duplicate_groups}")
display(duplicates_target.orderBy(F.desc("count")))

### Resultado esperado
La consulta anterior debería devolver **0 grupos duplicados** en la tabla final.

## 8. Detectar registros del bronze.unificado que no existen en silver.unificado_final

In [0]:
# Comparación por clave de negocio: ID, MUESTRA, RESULTADO
source_keys = source_df.select(*KEY_COLUMNS).dropDuplicates()
silver_unificado_final_keys = target_df.select(*KEY_COLUMNS).dropDuplicates()

missing_in_target = source_keys.join(
    silver_unificado_final_keys,
    on=KEY_COLUMNS,
    how="left_anti"
)

missing_count = missing_in_target.count()
print(f"Claves presentes en bronze.unificado pero ausentes en silver_unificado_final: {missing_count}")
display(missing_in_target.limit(100))

## 9. Verificar FECHA_COPIA

El challenge indica que `FECHA_COPIA` llega vacía en el archivo de entrada y debe completarse durante la inserción.

In [0]:
fecha_nulls = target_df.filter(F.col(DATE_COLUMN).isNull()).count()
print(f"silver.unificado_final - registros con {DATE_COLUMN} NULL: {fecha_nulls}")

print("Distribución por fecha:")
display(
    target_df.groupBy(DATE_COLUMN)
    .count()
    .orderBy(F.col(DATE_COLUMN).desc())
)

## 10. Validar que se conserva el último registro de cada duplicado

In [0]:
# Este control busca grupos duplicados en bronze.unificado y comprueba que silver.unificado_final 
# tenga un único registro para cada clave de negocio.
# Si bronze.unificado conserva histórico, silver.unificado_final debe contener una única fila por clave.

source_duplicate_keys = (
    source_df.groupBy(*KEY_COLUMNS)
    .count()
    .filter(F.col("count") > 1)
    .select(*KEY_COLUMNS)
)

target_for_source_duplicates = (
    target_df.join(source_duplicate_keys, on=KEY_COLUMNS, how="inner")
    .groupBy(*KEY_COLUMNS)
    .count()
)

still_duplicated = target_for_source_duplicates.filter(F.col("count") > 1)
print(f"Claves duplicadas en bronze.unificado que siguen duplicadas en silver.unificado_final: {still_duplicated.count()}")
display(still_duplicated)

## 11. Resumen automático de controles

Esta celda genera una tabla que se puede utilizar como evidencia en la documentación del challenge.

In [0]:
summary = [
    ("bronze.unificado tabla existe", source_exists),
    ("silver.unificado_final tabla existe", target_exists),
    ("Esquema compatible", schema_issues == 0),
    ("Conteo de claves de negocio distintas preservado", source_distinct_keys == target_distinct_keys),
    ("Sin grupos duplicados en silver.unificado_final", duplicate_groups == 0),
    ("Sin claves de negocio de bronze.unificado ausentes en silver.unificado_final", missing_count == 0),
    (f"Sin NULLs en {DATE_COLUMN} en silver.unificado_final", fecha_nulls == 0),
    ("Sin duplicados restantes de claves duplicadas en SOURCE", still_duplicated.count() == 0),
]

summary_df = spark.createDataFrame(summary, ["control", "aprobado"])
display(summary_df)

failed = summary_df.filter(~F.col("aprobado")).count()
print(f"Controles aprobados: {len(summary) - failed}/{len(summary)}")

if failed > 0:
    print("ADVERTENCIA: Hay controles que requieren revisión.")
else:
    print("OK: Todos los controles definidos fueron satisfactorios.")

# Conclusión

Los resultados de esta notebook deben utilizarse como evidencia de testing.

Antes de afirmar que no hubo pérdida de información, revisar especialmente:
1. cantidad de claves de negocio distintas,
2. claves presentes en bronze.unificado  pero ausentes en silver.unificado_final,
3. duplicados en silver.unificado_final,
4. NULLs inesperados,
5. consistencia de los tipos de datos y columnas,
6. correcta carga de `FECHA_COPIA`.

Si alguno de estos controles falla, investigar la causa antes de documentar el proceso como exitoso.